# Cross-Instrument Electrochemical Benchmarking
## Metrohm Autolab (NOVA) vs. BioLogic (EC-Lab)

This notebook executes a rigorous inter-instrument comparison between **Metrohm Autolab** and **BioLogic** potentiostats for the identical catalyst system under standardized experimental conditions (1.0 M KOH, scan rate $\nu = 5\,\mathrm{mV/s}$, $R_u = 4.2\,\Omega$, $A = 0.196\,\mathrm{cm}^2$).

### Objectives:
1. Demonstrate standardized data harmonization across disparate manufacturer formats.
2. Quantify kinetic agreement ($\eta_{10}, \eta_{50}$) and inter-instrument residual current densities $\Delta j(E)$.
3. Compare quasi-steady-state Tafel slopes and assess potential measurement bias.


In [ ]:
import sys
from pathlib import Path

# Add project root to sys.path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scripts.utils import apply_publication_style, create_publication_figure
from parsers.electrochemistry import load_electrochemistry_data
from modules.electrochemistry import (
    compare_instrument_lsv,
    plot_cross_instrument_benchmark,
    calculate_tafel_slope,
    extract_overpotential_at_current,
)

apply_publication_style()
data_dir = project_root / "data" / "examples" / "electrochemistry"
print("Cross-instrument benchmarking environment ready.")


## 1. Import and Align Instrument Datasets

Load matched OER polarization scans from both workstations.


In [ ]:
metrohm_file = data_dir / "metrohm" / "metrohm_oer_lsv.txt"
biologic_file = data_dir / "biologic" / "biologic_oer_lsv.mpt"

data_m = load_electrochemistry_data(metrohm_file, instrument="metrohm", technique="LSV")
data_b = load_electrochemistry_data(biologic_file, instrument="biologic", technique="LSV")

print(f"Metrohm Data Points: {len(data_m.potential_V)} | Area: {data_m.electrode_area_cm2} cm^2")
print(f"BioLogic Data Points: {len(data_b.potential_V)} | Area: {data_b.electrode_area_cm2} cm^2")


## 2. Cross-Instrument Polarization Benchmark & Residuals

Evaluate the residual difference $\Delta j(E) = j_{\mathrm{Metrohm}}(E) - j_{\mathrm{BioLogic}}(E)$ across the kinetic polarization window.


In [ ]:
comp = compare_instrument_lsv(data_m, data_b, reaction="OER", target_j=10.0)

print("--- Inter-Instrument Benchmark Statistics ---")
print(f"Conditions Match: {comp['conditions_match']}")
print(f"Metrohm eta_10:  {comp['eta10_metrohm_mV']:.1f} mV")
print(f"BioLogic eta_10: {comp['eta10_biologic_mV']:.1f} mV")
print(f"Delta eta_10:    {comp['delta_eta10_mV']:.2f} mV")
print(f"Current Density RMSE: {comp['rmse_current_density']:.3f} mA/cm^2")

fig, (ax_top, ax_bot) = plot_cross_instrument_benchmark(
    data_m,
    data_b,
    reaction="OER",
    title="Cross-Instrument OER Benchmark: Metrohm Autolab vs BioLogic",
)
plt.show()


## 3. Kinetic Parameters Concordance Table

Compare extracted overpotentials ($\eta_{10}, \eta_{50}$) and Tafel slopes between instruments.


In [ ]:
# Extract overpotentials
pot_m = data_m.get_potential_vs_rhe()
j_m = data_m.get_current_density(unit="mA/cm^2")
pot_b = data_b.get_potential_vs_rhe()
j_b = data_b.get_current_density(unit="mA/cm^2")

eta_m_10 = extract_overpotential_at_current(pot_m, j_m, 10.0, "OER")
eta_b_10 = extract_overpotential_at_current(pot_b, j_b, 10.0, "OER")
eta_m_20 = extract_overpotential_at_current(pot_m, j_m, 20.0, "OER")
eta_b_20 = extract_overpotential_at_current(pot_b, j_b, 20.0, "OER")

# Tafel Slopes in kinetic region (1.0 to 10.0 mA/cm^2)
tafel_m = calculate_tafel_slope(pot_m, j_m, current_range=(1.0, 10.0), reaction="OER", scan_rate_V_s=0.005)
tafel_b = calculate_tafel_slope(pot_b, j_b, current_range=(1.0, 10.0), reaction="OER", scan_rate_V_s=0.005)

summary_df = pd.DataFrame([
    {
        "Parameter": "eta_10 (mV)",
        "Metrohm Autolab": f"{eta_m_10:.1f}",
        "BioLogic EC-Lab": f"{eta_b_10:.1f}",
        "Absolute Delta": f"{abs(eta_m_10 - eta_b_10):.1f} mV"
    },
    {
        "Parameter": "eta_20 (mV)",
        "Metrohm Autolab": f"{eta_m_20:.1f}",
        "BioLogic EC-Lab": f"{eta_b_20:.1f}",
        "Absolute Delta": f"{abs(eta_m_20 - eta_b_20):.1f} mV"
    },
    {
        "Parameter": "Tafel Slope (mV/dec)",
        "Metrohm Autolab": f"{tafel_m['tafel_slope_mV_dec']:.1f}",
        "BioLogic EC-Lab": f"{tafel_b['tafel_slope_mV_dec']:.1f}",
        "Absolute Delta": f"{abs(tafel_m['tafel_slope_mV_dec'] - tafel_b['tafel_slope_mV_dec']):.1f} mV/dec"
    },
    {
        "Parameter": "Exchange Current j_0 (mA/cm^2)",
        "Metrohm Autolab": f"{tafel_m['exchange_current_density_mA_cm2']:.2e}",
        "BioLogic EC-Lab": f"{tafel_b['exchange_current_density_mA_cm2']:.2e}",
        "Absolute Delta": "-"
    }
])

print(summary_df.to_string(index=False))
